In [ ]:
# --- locate the package and the data root, then run from the data root ---
# Layout after moving the code into a subfolder:
#   <DATA_ROOT>/                       <- chdir here; config paths resolve here
#       train/  embeddings/  outputs/  checkpoints/
#       multiomic-missing-data-pipeline/
#           transformer_pipeline/      <- the importable package
#           experiments/               <- this notebook
# Walk up from the cwd to find each marker independently, so it works no matter
# where the kernel was launched. Idempotent: safe to re-run.
import os, sys


def _find_up(start, marker):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Could not locate '{marker}/' above {start!r}.")
        d = parent


# Package dir = the folder that *contains* `transformer_pipeline/`.
_PKG_PARENT = _find_up(os.getcwd(), "transformer_pipeline")
_PKG = os.path.join(_PKG_PARENT, "transformer_pipeline")
if _PKG not in sys.path:
    sys.path.insert(0, _PKG)

# Data root = the nearest folder that holds `train/` (the input CSVs).
_DATA_ROOT = _find_up(os.getcwd(), "train")
os.chdir(_DATA_ROOT)

print("cwd ->", os.getcwd())
print("pkg ->", _PKG)


# Stage 1 - Tokenizer A/B (scale vs add vs affine)

Each input feature value is fused with its node2vec embedding to form a token.
This notebook trains BOTH models under ALL THREE fusion modes and compares them.

```
scale   token = value * embedding                 (ORIGINAL)
add     token = embedding + slope * value          (pure additive)
affine  token = embedding * (1 + slope * value)    (entangled)
```

- **scale** - a feature at its mean (~0 after standardization) gives a ~zero
  token, so the encoder cannot see it. Suspected to cause mean-collapse.
- **add** - embedding (identity) always present; the value scales a SEPARATE
  learnable per-dim direction added on top. Value signal fully separable.
- **affine** - identity present but value stays entangled with the embedding
  direction. (In an earlier run this suppressed Model A's learning entirely -
  included here so you can confirm it for yourself.)

`slope` = a learnable per-dimension vector (`value_slope`), init 1.0.

**What to look for**
- Highest `peak_r` (and lowest Model A `val_mse`) wins; compare to the ridge
  ceiling (A ~0.12, B ~0.14).
- Watch `train_mse`: a mode that cannot drive *training* loss down is broken.
- Model B's `val_mse` is flat on the tiny simplex scale, so B is selected/judged
  by `per_feature_r` until Stage 2 (CLR) rescales the target.

Run from the repo root (folder containing `pipeline/`): Cell > Run All.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import pandas as pd
from pipeline.config import Config
from pipeline.pipeline import ImputationPipeline

EPOCHS = 60          # enough to see the curves separate; bump if you have time
PERMS  = 99          # Mantel is secondary now; keep it cheap
MODES  = ["scale", "add", "affine"]

def run(tokenizer, b_select="per_feature_r"):
    cfg = Config()
    cfg.model_a.tokenizer = tokenizer
    cfg.model_b.tokenizer = tokenizer
    cfg.train.max_epochs = EPOCHS
    cfg.train.patience   = EPOCHS        # no early stop -> see the full curve
    cfg.eval.mantel_permutations = PERMS
    pipe = ImputationPipeline(cfg)
    pipe.prepare_data()
    ta = pipe.train_model_a()            # Model A: val_mse selection (honest)
    cfg.train.selection_metric = b_select
    tb = pipe.train_model_b()            # Model B: per_feature_r (val_mse flat)
    return ta, tb

results = {}
for m in MODES:
    print(f"\n========== tokenizer = {m.upper()} ==========")
    results[m] = run(m)

### Summary: all three modes, per-feature r vs the ridge ceiling

In [ ]:
def selected(trainer):
    e = trainer.history.best_epoch
    rec = [r for r in trainer.history.records if r.epoch == e][0]
    return rec.per_feature_r, rec.val_mse

def peak_r(trainer):
    vals = [r.per_feature_r for r in trainer.history.records
            if r.per_feature_r is not None]
    return max(vals) if vals else float("nan")

def min_train_mse(trainer):
    return min(r.train_loss for r in trainer.history.records)

rows = []
for mi, (model_name, idx) in enumerate(
        [("Model A (micro->metab)", 0), ("Model B (metab->micro)", 1)]):
    for m in MODES:
        tr = results[m][idx]
        sel_r, sel_mse = selected(tr)
        rows.append({
            "model": model_name,
            "tokenizer": m,
            "peak_r":       round(peak_r(tr), 4),
            "selected_r":   round(sel_r, 4) if sel_r is not None else None,
            "selected_val_mse": round(sel_mse, 5),
            "min_train_mse":    round(min_train_mse(tr), 5),
        })

summary = pd.DataFrame(rows)
print("RIDGE CEILING (target):  A per_feat_r ~0.12   B per_feat_r ~0.14\n")
print(summary.to_string(index=False))

print("\nWinner by peak_r:")
for model_name in summary.model.unique():
    sub = summary[summary.model == model_name]
    best = sub.loc[sub.peak_r.idxmax()]
    print(f"  {model_name}: {best.tokenizer}  (peak_r={best.peak_r})")

### How to read this

- **Highest `peak_r`** is the best fusion for that model. Compare to the ridge
  ceiling (A ~0.12, B ~0.14) to see how much signal is still on the table.
- **`min_train_mse`** is the sanity check: if a mode can't drive *training* loss
  down, it's broken regardless of `peak_r` (this is how `affine` failed Model A -
  its `train_mse` stayed pinned near 1.0).
- Model A's improvement also shows as a lower **`selected_val_mse`**. Model B's
  `val_mse` stays flat on the simplex scale until Stage 2 (CLR).

Once a winner is clear, set it as the default and we move to Stage 2.

Terminal equivalent for a single mode:
```
python run.py \
  --micro-emb embeddings/aligned/microbe_embeddings.csv \
  --metab-emb embeddings/aligned/metabolite_embeddings.csv \
  --max-epochs 60 --tokenizer add --selection-metric val_mse --permutations 99
```